# Phase 1 -- Notebook p1_04: Model B trainer (local/global, resumable)

**Model B** is the long-context model: most layers only look at the
previous 512 tokens (sliding window), and every 6th layer looks at the
whole sequence. Local layers use RoPE, global layers use no position
encoding (iRoPE-style). One shared key/value head (MQA) keeps the
generation cache tiny. ~122M parameters, tokenizer T1.

**How to run it**
1. Attach inputs: `legallm-phase1-data`, `legallm-tokenizers`.
   From the second main session on, also attach `legallm-checkpoints-b`
   (this notebook's own checkpoint dataset -- **not** A's `legallm-checkpoints`).
2. Add-ons -> Secrets: `KAGGLE_API_TOKEN` switched on. Internet on. GPU T4 x2.
3. First time: `SMOKE_MODE = True`, plain **Run All** (about 25-30 min) and
   check it reaches the end with no errors.
4. Then `SMOKE_MODE = False` -> **Save Version -> Save & Run All**.

**Two modes, one code path** (Section 2): smoke = 15-minute pipeline
check; main = the real run, resumable across sessions.


## 1. Imports, GPUs, and input folders

Input folders are found by searching `/kaggle/input` for a file each
dataset is known to contain, instead of hard-coding one path. Kaggle has
mounted attached datasets at more than one depth, and a hard-coded path
fails with a bare "No such file or directory".


In [ ]:
import datetime
import gc
import itertools
import json
import math
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("\nWARNING: no GPU detected. Set Accelerator -> GPU T4 x2 in the notebook settings.")

KAGGLE_INPUT = Path("/kaggle/input")


def find_input_dir(marker, dataset_label):
    """Folder that contains `marker`, searched anywhere under /kaggle/input."""
    hits = sorted(KAGGLE_INPUT.rglob(marker)) if KAGGLE_INPUT.exists() else []
    if not hits:
        attached = sorted(p.name for p in KAGGLE_INPUT.glob("*")) if KAGGLE_INPUT.exists() else []
        raise FileNotFoundError(
            f"Could not find '{marker}' under /kaggle/input. Attach the '{dataset_label}' dataset "
            f"(right panel -> Input -> Add Input) and re-run this cell. Attached right now: {attached}"
        )
    return hits[0].parent


INPUT_DATA_DIR = find_input_dir("dev_prompts.json", "legallm-phase1-data")
INPUT_TOKENIZER_DIR = find_input_dir("t1_bpe.json", "legallm-tokenizers")

OUTPUT_DIR = Path("/kaggle/working/phase1_model_b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Flat on purpose: checkpoints sit directly in OUTPUT_DIR. A sub-folder gets
# uploaded by the Kaggle CLI as a .zip, which makes resume harder to find.
CHECKPOINT_DIR = OUTPUT_DIR

print(f"\nData folder:      {INPUT_DATA_DIR}")
print(f"Tokenizer folder: {INPUT_TOKENIZER_DIR}")
print(f"Output folder:    {OUTPUT_DIR}")


## 2. Run mode & config

The only lines you normally touch: `SMOKE_MODE`, and in main mode
`TIME_BUDGET_MIN` (this session) and `TOTAL_TRAIN_HOURS` (the whole B run,
across all sessions -- the learning-rate schedule is sized to it).

Default main mode is **one 4-hour session** that runs the full schedule
(warmup -> decay to the minimum LR), so the checkpoint at the end is a
finished model even if you never get to run a second session. If you have
quota for two sessions, set `TOTAL_TRAIN_HOURS = 8` **before the first main
session** (the schedule is fixed at the first main session and saved in the
checkpoint).


In [ ]:
SMOKE_MODE = True  # <-- True: 15-min pipeline check.  False: the real run.

MODEL_B_CONFIG = dict(
    vocab_size=32_000,        # T1 (BPE)
    d_model=640,
    n_layer=24,               # deep-narrow
    n_head=10,
    n_kv_head=1,              # MQA: one shared key/value head
    d_head=64,
    d_ff=1728,
    max_seq_len=2048,
    window=512,               # sliding-window size for local layers
    global_every=6,           # every 6th layer is global (1 global per 5 local)
    qk_norm=True,
    tokenizer_id="T1",
    attn_type="local+global",
    tied_embeddings=True,
)
ARCH_ID = "B"
# Smoke checkpoints get a different file prefix, so a main run can never
# accidentally resume from a smoke checkpoint (its LR has already decayed).
CKPT_PREFIX = "Bsmoke" if SMOKE_MODE else ARCH_ID

USE_BOTH_GPUS = True            # nn.DataParallel over both T4s (auto-fallback to 1 GPU, Section 5)
PREFERRED_SEQ_LENS = [2048, 1024]  # try B's designed context first; Section 5 picks what fits
TOKENS_PER_STEP_TARGET = 32_768    # same tokens per optimizer step as Model A -> comparable runs

# 6e-4 is what GPT-3 used at ~125M params. B has QK-norm + z-loss for fp16
# stability, and short runs benefit from a higher peak LR than A's 3e-4.
PEAK_LR = 6e-4
MIN_LR = PEAK_LR / 10
WEIGHT_DECAY = 0.1
GRAD_CLIP_NORM = 1.0
Z_LOSS_COEF = 1e-4              # keeps output logits from drifting large (fp16 safety)
LR_DECAY_STEPS_OVERRIDE = None  # leave None; set an int only to force a new schedule on resume

if SMOKE_MODE:
    TIME_BUDGET_MIN = 15
    TOTAL_TRAIN_HOURS = 0.25
    CHECKPOINT_EVERY_MIN = 6        # two checkpoints in 15 min -> save + prune both get exercised
    PUSH_DURING_TRAINING = False    # smoke pushes once, at the end (Section 15)
else:
    TIME_BUDGET_MIN = 240           # this session's training time
    TOTAL_TRAIN_HOURS = 4.0         # the whole B run; >= TIME_BUDGET_MIN / 60
    CHECKPOINT_EVERY_MIN = 45
    PUSH_DURING_TRAINING = True

assert TIME_BUDGET_MIN <= 11 * 60, "Keep each session under Kaggle's 12h cap (leave room for eval + push)."
assert TOTAL_TRAIN_HOURS * 60 >= TIME_BUDGET_MIN, "TOTAL_TRAIN_HOURS must cover at least this session."

print(f"SMOKE_MODE = {SMOKE_MODE}   (checkpoint prefix '{CKPT_PREFIX}')")
print(f"Model B config: {json.dumps(MODEL_B_CONFIG, indent=2)}")
print(f"TIME_BUDGET_MIN={TIME_BUDGET_MIN}  TOTAL_TRAIN_HOURS={TOTAL_TRAIN_HOURS}  "
      f"CHECKPOINT_EVERY_MIN={CHECKPOINT_EVERY_MIN}  PEAK_LR={PEAK_LR}")


## 3. Tokenizer

T1 (byte-level BPE, 32k), same as Model A.


In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(INPUT_TOKENIZER_DIR / "t1_bpe.json"))
EOS_ID = tokenizer.token_to_id("<eos>")
assert EOS_ID is not None, "T1 tokenizer is missing the <eos> special token."
assert tokenizer.get_vocab_size() <= MODEL_B_CONFIG["vocab_size"], "Tokenizer vocab is larger than the model's embedding table."

print(f"Tokenizer vocab size: {tokenizer.get_vocab_size()}")
print(f"<eos> id: {EOS_ID}")
print("Round trip check:", tokenizer.decode(tokenizer.encode("Section 302 of the Indian Penal Code.").ids))


## 4. The model

Same building blocks and method names as Model A (`encode`, `forward`,
`generate`, `build_model`; named `q/k/v/o_proj`, `gate/up/down_proj`), so
the PEFT/KD library in later phases works on it unchanged. What's
different:

| Piece | Model A | Model B |
|---|---|---|
| Attention span | full (causal) in every layer | 512-token window in local layers; full in every 6th (global) layer |
| Position encoding | RoPE everywhere | RoPE in local layers, none (NoPE) in global layers |
| KV heads | 2 (GQA) | 1 (MQA) |
| Shape | 16 layers x 896 | 24 layers x 640 |
| QK-norm | no | yes (normalizes q and k per head -- fp16 stability) |

**How the window works:** a boolean mask `[query, key]` that is True only
when `0 <= query_pos - key_pos < 512`, passed to PyTorch's
`scaled_dot_product_attention`. The same mask function is used in training
and in cached generation, so the two can't disagree -- the self-test at
the end of this cell checks exactly that.

**Loss is computed inside `forward()`** (when `targets` is given). That's
what lets `nn.DataParallel` split a batch across both GPUs without first
copying the full `[batch, seq, 32000]` logits back to GPU 0.


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :].to(x.dtype)
    sin = sin[None, None, :, :].to(x.dtype)
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


_TRAIN_MASK_CACHE = {}


def sliding_window_mask(q_start, n_q, n_k, window, device, cacheable=False):
    """Boolean [n_q, n_k] mask, True = may attend. Query i sits at absolute
    position q_start + i; key j at position j. Allowed iff the key is not in
    the future AND is less than `window` positions back.
    Only training masks are cached (training always uses one fixed shape);
    eval/generation shapes vary per prompt, so caching them would leak memory.
    """
    cache_key = (n_q, n_k, window, str(device))
    if cacheable and cache_key in _TRAIN_MASK_CACHE:
        return _TRAIN_MASK_CACHE[cache_key]
    q_pos = torch.arange(q_start, q_start + n_q, device=device)[:, None]
    k_pos = torch.arange(n_k, device=device)[None, :]
    allowed = (k_pos <= q_pos) & (q_pos - k_pos < window)
    if cacheable:
        _TRAIN_MASK_CACHE[cache_key] = allowed
    return allowed


class LocalGlobalAttention(nn.Module):
    """MQA attention. Local layers: RoPE + sliding-window mask. Global
    layers: no RoPE (NoPE), plain causal attention over everything.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, use_rope, qk_norm):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.use_rope = use_rope
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)
        self.q_norm = RMSNorm(d_head) if qk_norm else None
        self.k_norm = RMSNorm(d_head) if qk_norm else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.use_rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        # The cache keeps the single KV head (that's MQA's memory saving);
        # expand it to all query heads only for the attention call itself.
        # An explicit copy keeps SDPA on its fast memory-efficient kernel.
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            # Global layer. Training / prefill: square causal. Decode step
            # (one query): may see every cached key -> no mask needed.
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    def __init__(self, cfg, is_global):
        super().__init__()
        self.is_global = is_global
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = LocalGlobalAttention(cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"],
                                         use_rope=not is_global, qk_norm=cfg["qk_norm"])
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg["d_model"], cfg["d_ff"])

    def _forward_inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, attn_mask, cache)


class KVCache:
    """Pre-allocated per-layer cache. Keeps the full history (with 1 KV
    head it's only ~0.5 MB per layer at 2048 tokens); the sliding window is
    applied by the attention mask, not by evicting entries.
    """
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


def is_global_layer(layer_idx, global_every):
    return (layer_idx + 1) % global_every == 0


class LocalGlobalGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config, is_global=is_global_layer(i, config["global_every"]))
            for i in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # tied embeddings

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def layer_pattern(self):
        return "".join("G" if b.is_global else "L" for b in self.blocks)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def _run_blocks(self, x, pos_offset, caches, use_ckpt):
        T = x.shape[1]
        assert pos_offset + T <= self.config["max_seq_len"], "Sequence longer than max_seq_len."
        cos = self.rope_cos[pos_offset:pos_offset + T]
        sin = self.rope_sin[pos_offset:pos_offset + T]
        local_mask = sliding_window_mask(pos_offset, T, pos_offset + T, self.config["window"], x.device,
                                         cacheable=self.training and caches is None)
        for i, blk in enumerate(self.blocks):
            mask = None if blk.is_global else local_mask
            cache = caches[i] if caches is not None else None
            x = blk(x, cos, sin, mask, cache=cache, use_checkpoint=use_ckpt)
        return x

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: hidden states, no LM head. attention_mask is
        accepted for interface compatibility (Phase 1 packs full windows)."""
        assert mode == "causal", "Model B is decoder-only; only causal encoding is supported."
        x = self.embed(input_ids)
        x = self._run_blocks(x, 0, None, use_ckpt=self.training and USE_GRAD_CHECKPOINTING)
        return self.norm_f(x)

    def forward(self, ids, targets=None, caches=None):
        """No targets -> logits. With targets -> tensor [1, 2] = [cross-entropy,
        cross-entropy + z-loss]. The [1, 2] shape is so nn.DataParallel can
        stack one row per GPU; callers take .mean(0).
        In training mode, fp16 autocast is switched on here (inside the
        module) so it also applies inside DataParallel's worker threads.
        """
        use_amp = self.training and ids.is_cuda
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            if caches is None:
                h = self.encode(ids)
            else:
                pos_offset = caches[0].pos
                assert ids.shape[1] == 1 or pos_offset == 0, "Prefill must start from an empty cache."
                h = self.norm_f(self._run_blocks(self.embed(ids), pos_offset, caches, use_ckpt=False))
            logits = self.head(h)
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        target_logit = logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        ce = (lse - target_logit).mean()           # identical to F.cross_entropy
        total = ce + Z_LOSS_COEF * lse.pow(2).mean()
        return torch.stack([ce, total]).unsqueeze(0)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy",
                 top_p=0.9, temperature=1.0, repetition_penalty=1.0):
        """Contract method. Stops per sequence at <eos>. Prompts too long to
        leave room for max_new_tokens are cut to their most recent tokens."""
        assert kv_cache, "Only the KV-cache path is implemented."
        was_training = self.training
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])
        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)[:, -1, :]
        next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)
        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all() or generated.shape[1] >= max_total:
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)[:, -1, :]
            next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)

        self.train(was_training)
        return generated

    @staticmethod
    def _penalize(logits, previous_ids, penalty):
        """CTRL-style repetition penalty on every token already present."""
        if penalty == 1.0:
            return logits
        scores = logits.gather(1, previous_ids)
        scores = torch.where(scores > 0, scores / penalty, scores * penalty)
        return logits.scatter(1, previous_ids, scores)

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        keep = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * keep
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function."""
    return LocalGlobalGPT(config)


USE_GRAD_CHECKPOINTING = True


def _model_self_test():
    """Seconds-long correctness checks on a tiny config, before spending
    GPU-hours: (1) cached generation gives the same logits as a full
    forward pass, with the sequence longer than the window and global
    layers present; (2) a local-only layer really cannot see past its
    window; (3) the in-forward loss equals F.cross_entropy."""
    torch.manual_seed(0)
    tiny = dict(MODEL_B_CONFIG, vocab_size=101, d_model=64, n_layer=6, n_head=4, n_kv_head=1,
                d_head=16, d_ff=128, max_seq_len=64, window=8, global_every=3)
    m = build_model(tiny).to(DEVICE).eval()
    ids = torch.randint(0, 101, (2, 40), device=DEVICE)
    with torch.no_grad():
        full = m(ids)
        caches = m.new_kv_caches(2, DEVICE, torch.float32)
        parts = [m(ids[:, :12], caches=caches)]
        for t in range(12, 40):
            parts.append(m(ids[:, t:t + 1], caches=caches))
        stepped = torch.cat(parts, dim=1)
    diff = (full - stepped).abs().max().item()
    assert diff < 1e-3, f"KV-cache logits differ from full forward by {diff}"

    local_only = build_model(dict(tiny, n_layer=1, global_every=10 ** 6)).to(DEVICE).eval()
    ids2 = ids.clone()
    ids2[:, 0] = (ids[:, 0] + 1) % 101
    with torch.no_grad():
        a, b = local_only(ids), local_only(ids2)
    far = (a[:, 8:] - b[:, 8:]).abs().max().item()     # positions >= window: token 0 out of view
    near = (a[:, 7] - b[:, 7]).abs().max().item()      # position window-1: token 0 in view
    assert far < 1e-5 and near > 1e-5, f"Sliding window leak/too-narrow: far={far}, near={near}"

    targets = torch.randint(0, 101, (2, 40), device=DEVICE)
    with torch.no_grad():
        ce_inside = m(ids, targets=targets)[0, 0].item()
        ce_ref = F.cross_entropy(m(ids).reshape(-1, 101), targets.reshape(-1)).item()
    assert abs(ce_inside - ce_ref) < 1e-4, f"In-forward loss {ce_inside} != F.cross_entropy {ce_ref}"
    print(f"Self-test passed: cache-vs-full diff={diff:.1e}, window far={far:.1e} near={near:.1e}, "
          f"loss match {ce_inside:.4f}={ce_ref:.4f}")


_model_self_test()
_TRAIN_MASK_CACHE.clear()

torch.manual_seed(1234)
model = build_model(MODEL_B_CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model B parameter count (tied embeddings): {n_params / 1e6:.1f}M")
print(f"Layer pattern (L=local, G=global): {model.layer_pattern()}")
assert 50e6 <= n_params <= 300e6, "outside the competition's allowed parameter range!"


## 5. Two GPUs, batch size, sequence length, and the LR schedule

**Two GPUs.** Kaggle charges the same quota for T4 x2 whether you use one
GPU or both. `nn.DataParallel` splits each batch across the two GPUs inside
this one notebook process (no launcher needed). A quick self-test compares
its loss to single-GPU on the same batch and checks gradients arrive; if
anything is off, it falls back to one GPU automatically.

**What fits.** Tries `seq_len=2048` (B's designed context) first, then
1024, at increasing batch sizes, with real optimizer memory. The weights
are restored afterwards so the probe steps don't leak into training.

**Schedule.** The probe also times each step. From that, the notebook
works out how many optimizer steps `TOTAL_TRAIN_HOURS` buys and sizes the
cosine decay to end there -- so the run ends on a fully decayed learning
rate instead of stopping halfway down the curve.


In [ ]:
def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda")
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler()


def reset_gpu_memory():
    gc.collect()
    for d in range(N_GPUS):
        with torch.cuda.device(d):
            torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(d)


def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()


def make_optimizer(module, lr):
    """AdamW with weight decay on matrices only (not on norm gains --
    decaying QK-norm gains would cap how sharp attention can get)."""
    decay = [p for p in module.parameters() if p.dim() >= 2]
    no_decay = [p for p in module.parameters() if p.dim() < 2]
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.95), eps=1e-8,
    )


# --- Two-GPU setup with self-test --------------------------------------------
train_model, USING_DP, GPUS_USED = model, False, 1
if USE_BOTH_GPUS and N_GPUS >= 2:
    try:
        dp_model = nn.DataParallel(model, device_ids=[0, 1])
        model.train()
        x_t = torch.randint(0, MODEL_B_CONFIG["vocab_size"], (4, 256), device=DEVICE)
        y_t = torch.randint(0, MODEL_B_CONFIG["vocab_size"], (4, 256), device=DEVICE)
        with torch.no_grad():
            single = model(x_t, targets=y_t).mean(0)
            multi = dp_model(x_t, targets=y_t).mean(0)
        loss_gap = (single - multi).abs().max().item()
        dp_model(x_t, targets=y_t).mean(0)[1].backward()
        grads_ok = all(p.grad is not None and torch.isfinite(p.grad).all().item() for p in model.parameters())
        model.zero_grad(set_to_none=True)
        if loss_gap < 2e-2 and grads_ok:
            train_model, USING_DP, GPUS_USED = dp_model, True, 2
            print(f"DataParallel self-test passed (loss gap {loss_gap:.1e}) -> training on BOTH GPUs.")
        else:
            print(f"DataParallel self-test failed (loss gap {loss_gap:.1e}, grads ok={grads_ok}) -> 1 GPU.")
    except Exception as exc:
        model.zero_grad(set_to_none=True)
        print(f"DataParallel unavailable ({type(exc).__name__}: {exc}) -> training on 1 GPU.")
else:
    print(f"Training on 1 GPU (GPUs visible: {N_GPUS}, USE_BOTH_GPUS={USE_BOTH_GPUS}).")

# --- Batch-size / sequence-length probe ---------------------------------------
MEMORY_BUDGET_GB = 13.0   # per GPU, of the T4's ~15 GB usable
CANDIDATE_BATCH_SIZES = [b for b in [2, 4, 6, 8, 12, 16, 24, 32] if b % GPUS_USED == 0]
initial_state = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}


def try_config(batch_size, seq_len, n_iters=3):
    """Returns (peak_gb, seconds_per_micro_step) or (None, None) on OOM."""
    reset_gpu_memory()
    probe_optim = None
    try:
        probe_optim = make_optimizer(model, 1e-4)
        probe_scaler = make_grad_scaler()
        x = torch.randint(0, MODEL_B_CONFIG["vocab_size"], (batch_size, seq_len), device=DEVICE)
        y = torch.randint(0, MODEL_B_CONFIG["vocab_size"], (batch_size, seq_len), device=DEVICE)
        timings = []
        for _ in range(n_iters):
            torch.cuda.synchronize()
            t0 = time.time()
            probe_optim.zero_grad(set_to_none=True)
            out = train_model(x, targets=y).mean(0)
            probe_scaler.scale(out[1]).backward()
            probe_scaler.step(probe_optim)
            probe_scaler.update()
            torch.cuda.synchronize()
            timings.append(time.time() - t0)
        peak_gb = max(torch.cuda.max_memory_allocated(d) for d in range(GPUS_USED)) / 1e9
        return peak_gb, min(timings[1:])   # first iteration includes warm-up
    except Exception as exc:
        if is_oom(exc):
            return None, None
        raise
    finally:
        del probe_optim
        model.zero_grad(set_to_none=True)
        reset_gpu_memory()


model.train()
probe_results = {}
for seq_len in PREFERRED_SEQ_LENS:
    for bs in CANDIDATE_BATCH_SIZES:
        if bs * seq_len > TOKENS_PER_STEP_TARGET:
            break   # bigger micro-batches than one optimizer step's worth of tokens aren't useful
        peak_gb, sec = try_config(bs, seq_len)
        if peak_gb is None:
            print(f"  seq_len={seq_len:5d} batch={bs:3d} -> OOM")
            break
        fits = peak_gb <= MEMORY_BUDGET_GB
        print(f"  seq_len={seq_len:5d} batch={bs:3d} -> {peak_gb:5.2f} GB/GPU, "
              f"{sec:.2f}s/micro-step, {bs * seq_len / sec:,.0f} tok/s" + ("" if fits else "  [over budget]"))
        if not fits:
            break
        probe_results[(seq_len, bs)] = sec
    fitting_here = [b for (s, b) in probe_results if s == seq_len]
    if fitting_here and max(fitting_here) * seq_len >= 8192:
        break   # longest context that trains efficiently -- no need to probe shorter ones

model.load_state_dict(initial_state)   # undo the probe's random-data updates
del initial_state
reset_gpu_memory()

# Longest context whose largest fitting batch still gives >= 8k tokens per micro-step.
chosen = None
for seq_len in PREFERRED_SEQ_LENS:
    fitting = [bs for (s, bs) in probe_results if s == seq_len]
    if fitting and max(fitting) * seq_len >= 8192:
        chosen = (seq_len, max(fitting))
        break
if chosen is None and probe_results:
    chosen = max(probe_results, key=lambda k: k[0] * k[1])
assert chosen is not None, "Nothing fit in memory -- lower MEMORY_BUDGET_GB assumptions or PREFERRED_SEQ_LENS."

TRAIN_SEQ_LEN, MICRO_BATCH_SIZE = chosen
SEC_PER_MICRO_STEP = probe_results[chosen]
GRAD_ACCUM_STEPS = 1 if SMOKE_MODE else max(1, round(TOKENS_PER_STEP_TARGET / (MICRO_BATCH_SIZE * TRAIN_SEQ_LEN)))
TOKENS_PER_STEP = MICRO_BATCH_SIZE * TRAIN_SEQ_LEN * GRAD_ACCUM_STEPS

# 1.2x safety margin for data loading + checkpoint pushes, so the decay
# finishes a little early (then holds at MIN_LR) rather than late.
est_sec_per_step = SEC_PER_MICRO_STEP * GRAD_ACCUM_STEPS * 1.2
planned_steps = max(50, int(TOTAL_TRAIN_HOURS * 3600 / est_sec_per_step))
SCHEDULE = {
    "peak_lr": PEAK_LR,
    "min_lr": MIN_LR,
    "warmup_steps": min(200, max(10, planned_steps // 20)),
    "lr_decay_steps": planned_steps,
    "tokens_per_step": TOKENS_PER_STEP,
    "total_train_hours": TOTAL_TRAIN_HOURS,
}

print(f"\nGPUs used: {GPUS_USED}  |  seq_len={TRAIN_SEQ_LEN}  micro-batch={MICRO_BATCH_SIZE}  "
      f"grad-accum={GRAD_ACCUM_STEPS}  -> {TOKENS_PER_STEP:,} tokens/step")
print(f"Measured ~{MICRO_BATCH_SIZE * TRAIN_SEQ_LEN / SEC_PER_MICRO_STEP:,.0f} tokens/s "
      f"-> ~{planned_steps:,} steps (~{planned_steps * TOKENS_PER_STEP / 1e6:,.0f}M tokens) "
      f"in {TOTAL_TRAIN_HOURS}h")
print(f"Schedule: {SCHEDULE}")


## 6. Checkpoint save/load (contract functions)

Same contract as Model A: `save_checkpoint(path, model, optim, sched,
scaler, step, tokens_seen, config)`, dict keys `{model_state, optim_state,
scheduler_state, scaler_state, step, tokens_seen, rng_state, config,
arch_id}`, file name `{arch_id}_step{step}_tokens{tokens_seen}.pt`.
B adds three extra keys: `schedule` (so resumed sessions keep the same LR
curve), `data_state` (where in each text file training got to), and
`run_mode`. Saves go to a temp file first and are then renamed, so a
session killed mid-save never leaves a half-written checkpoint behind.


In [ ]:
def save_checkpoint(path, model, optim, sched, scaler, step, tokens_seen, config, extra=None):
    state = {
        "model_state": model.state_dict(),
        "optim_state": optim.state_dict(),
        "scheduler_state": sched.state_dict() if sched is not None else None,
        "scaler_state": scaler.state_dict(),
        "step": step,
        "tokens_seen": tokens_seen,
        "rng_state": torch.get_rng_state(),
        "config": config,
        "arch_id": ARCH_ID,
    }
    if extra:
        state.update(extra)
    tmp_path = path.with_name(path.name + ".tmp")
    torch.save(state, tmp_path)
    tmp_path.replace(path)


def load_checkpoint(path, map_location):
    return torch.load(path, map_location=map_location, weights_only=False)  # our own files


def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def checkpoint_path_for(step, tokens_seen):
    return CHECKPOINT_DIR / f"{CKPT_PREFIX}_step{step}_tokens{tokens_seen}.pt"


def find_latest_checkpoint(prefix):
    """Newest '{prefix}_step*.pt' anywhere under /kaggle/input (any depth).
    Fallback: the file sitting inside a .zip (the Kaggle CLI uploads
    sub-folders as zips) -- only the newest member is extracted."""
    if not KAGGLE_INPUT.exists():
        return None
    pts = list(KAGGLE_INPUT.rglob(f"{prefix}_step*.pt"))
    if pts:
        return max(pts, key=step_of)
    best = None
    for zpath in KAGGLE_INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zpath) as zf:
                for name in zf.namelist():
                    base = Path(name).name
                    if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                        if best is None or step_of(base) > best[0]:
                            best = (step_of(base), zpath, name)
        except zipfile.BadZipFile:
            continue
    if best is None:
        return None
    _, zpath, name = best
    dest = CHECKPOINT_DIR / Path(name).name
    with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
        shutil.copyfileobj(src, dst)
    print(f"Extracted {Path(name).name} from {zpath.name}")
    return dest


print("save_checkpoint / load_checkpoint / find_latest_checkpoint defined.")


## 7. Kaggle Dataset push helper

Same as Model A, pointed at **B's own dataset** `legallm-checkpoints-b`.
(A new dataset version replaces the whole file list -- pushing B into
`legallm-checkpoints` would remove A's checkpoints from the latest version.)
Keeps the 2 newest local checkpoints before each push, so uploads stay
bounded (~1.5 GB each for B).


In [ ]:
from kaggle_secrets import UserSecretsClient

KAGGLE_USERNAME = "jatiniitk"
CHECKPOINT_DATASET_SLUG = "legallm-checkpoints-b"


def get_kaggle_token():
    try:
        token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    except Exception as exc:
        print(f"Could not read the 'KAGGLE_API_TOKEN' secret ({exc}). "
              f"Add-ons -> Secrets -> switch it on for this notebook.")
        return None
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    token_path = kaggle_dir / "access_token"
    token_path.write_text(token)
    token_path.chmod(0o600)
    return token


def prune_old_checkpoints(keep=2):
    ckpts = sorted(CHECKPOINT_DIR.glob(f"{CKPT_PREFIX}_step*.pt"), key=step_of)
    for p in ckpts[:-keep] if len(ckpts) > keep else []:
        p.unlink()


def push_checkpoint_dataset(message):
    if get_kaggle_token() is None:
        print("Skipping checkpoint push: no KAGGLE_API_TOKEN secret.")
        return False
    dataset_id = f"{KAGGLE_USERNAME}/{CHECKPOINT_DATASET_SLUG}"
    metadata = {"title": CHECKPOINT_DATASET_SLUG, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}
    with open(OUTPUT_DIR / "dataset-metadata.json", "w") as f:
        json.dump(metadata, f)
    exists = subprocess.run(["kaggle", "datasets", "status", dataset_id],
                            capture_output=True, text=True).returncode == 0
    if exists:
        cmd = ["kaggle", "datasets", "version", "-p", str(OUTPUT_DIR), "-m", message, "-r", "zip"]
    else:
        print(f"Dataset {dataset_id} does not exist yet -> creating it (private).")
        cmd = ["kaggle", "datasets", "create", "-p", str(OUTPUT_DIR), "-r", "zip"]
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Checkpoint push failed: {(result.stderr or result.stdout).strip()[-500:]}")
        return False
    print(f"Pushed checkpoint dataset in {time.time() - t0:.0f}s ({message}).")
    return True


print(f"Push target: {KAGGLE_USERNAME}/{CHECKPOINT_DATASET_SLUG}")


## 8. Optimizer, schedule, and resume

Looks for the newest `B_step*.pt` in any attached input (normally
`legallm-checkpoints-b`). If found: restores weights, optimizer, scaler,
RNG, step count, **the saved LR schedule**, and **where in each text file
training had got to**. Smoke checkpoints (`Bsmoke_...`) are never resumed
by a main run.


In [ ]:
optimizer = make_optimizer(model, PEAK_LR)
scaler = make_grad_scaler()


def get_lr(step):
    """Linear warmup, cosine decay to min_lr at lr_decay_steps, then flat."""
    s = SCHEDULE
    if step < s["warmup_steps"]:
        return s["peak_lr"] * (step + 1) / s["warmup_steps"]
    progress = min(1.0, (step - s["warmup_steps"]) / max(1, s["lr_decay_steps"] - s["warmup_steps"]))
    return s["min_lr"] + (s["peak_lr"] - s["min_lr"]) * 0.5 * (1 + math.cos(math.pi * progress))


start_step, start_tokens_seen, resume_file_positions = 0, 0, {}
latest = find_latest_checkpoint(CKPT_PREFIX)
if latest is None:
    print(f"No {CKPT_PREFIX}_step*.pt checkpoint in the attached inputs -- starting fresh "
          f"(expected for the first {'smoke' if SMOKE_MODE else 'main'} run).")
else:
    print(f"Resuming from {latest}")
    ckpt = load_checkpoint(latest, map_location="cpu")
    assert ckpt["config"] == MODEL_B_CONFIG, "Checkpoint config does not match MODEL_B_CONFIG!"
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    torch.set_rng_state(ckpt["rng_state"])
    start_step, start_tokens_seen = ckpt["step"], ckpt["tokens_seen"]
    resume_file_positions = ckpt.get("data_state", {}).get("file_positions", {})
    if ckpt.get("schedule"):
        SCHEDULE = ckpt["schedule"]
        print("Using the LR schedule saved in the checkpoint (fixed at the first main session).")
    if latest.parent != CHECKPOINT_DIR:
        shutil.copy(latest, CHECKPOINT_DIR / latest.name)   # keep it as the backup in the next push
    del ckpt
    gc.collect()
    print(f"Resumed at step={start_step:,}, tokens_seen={start_tokens_seen:,}")

if LR_DECAY_STEPS_OVERRIDE is not None:
    SCHEDULE = dict(SCHEDULE, lr_decay_steps=int(LR_DECAY_STEPS_OVERRIDE))
    print(f"LR_DECAY_STEPS_OVERRIDE -> lr_decay_steps={SCHEDULE['lr_decay_steps']}")

print(f"LR now: {get_lr(start_step):.2e}  (schedule: warmup {SCHEDULE['warmup_steps']}, "
      f"decay to step {SCHEDULE['lr_decay_steps']:,})")


## 9. Data pipeline

Same round-robin stream over the four cleaned text files as Model A, with
one addition: the stream **remembers its byte position in each file**, and
that position is saved in every checkpoint. A resumed session continues on
text it hasn't trained on yet, instead of re-reading the start of each file.


In [ ]:
class TokenStreamDataset(torch.utils.data.IterableDataset):
    """Yields (input_ids, target_ids) windows of seq_len tokens from a
    round-robin over `file_paths`, documents separated by <eos>. Loops
    forever; `self.positions` tracks the byte offset reached in each file.
    Must run with num_workers=0 (positions live in this process).
    """
    def __init__(self, file_paths, tokenizer, seq_len, start_positions=None):
        self.file_paths = [p for p in file_paths if p.stat().st_size > 0]
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        start_positions = start_positions or {}
        self.positions = {p.name: int(start_positions.get(p.name, 0)) for p in self.file_paths}
        self.wraps = {p.name: 0 for p in self.file_paths}

    def __iter__(self):
        buffer = []
        readers = [self._read_lines(p) for p in self.file_paths]
        for reader in itertools.cycle(readers):
            ids = self.tokenizer.encode(next(reader)).ids
            if not ids:
                continue
            buffer.extend(ids)
            buffer.append(EOS_ID)
            while len(buffer) >= self.seq_len + 1:
                window = buffer[:self.seq_len + 1]
                buffer = buffer[self.seq_len + 1:]
                yield torch.tensor(window[:-1], dtype=torch.long), torch.tensor(window[1:], dtype=torch.long)

    def _read_lines(self, path):
        name, size = path.name, path.stat().st_size
        while True:
            start = self.positions[name] if self.positions[name] < size else 0
            with open(path, "rb") as f:
                f.seek(start)
                pos = start
                while True:
                    raw = f.readline()
                    if not raw:
                        break
                    pos += len(raw)
                    self.positions[name] = pos
                    line = raw.decode("utf-8", errors="ignore").strip()
                    if line:
                        yield line
            self.positions[name] = 0
            self.wraps[name] += 1


TRAIN_FILES = sorted(INPUT_DATA_DIR.glob("clean_*.txt"))
assert TRAIN_FILES, f"No clean_*.txt files in {INPUT_DATA_DIR} -- check the attached legallm-phase1-data."
print("Training on:", [p.name for p in TRAIN_FILES])
if resume_file_positions:
    print("Continuing from saved file positions:",
          {k: f"{v / 1e6:.0f}MB" for k, v in resume_file_positions.items()})

train_dataset = TokenStreamDataset(TRAIN_FILES, tokenizer, TRAIN_SEQ_LEN, resume_file_positions)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=MICRO_BATCH_SIZE, num_workers=0)

# Shape check on a separate copy, so it doesn't move the real stream forward.
_check = TokenStreamDataset(TRAIN_FILES, tokenizer, TRAIN_SEQ_LEN, dict(train_dataset.positions))
_x, _y = next(iter(torch.utils.data.DataLoader(_check, batch_size=MICRO_BATCH_SIZE)))
assert torch.equal(_x[:, 1:], _y[:, :-1]), "targets must be inputs shifted by one"
print(f"Sample batch: x={tuple(_x.shape)}, y={tuple(_y.shape)}")
del _check, _x, _y


## 10. Training loop

Stops when `TIME_BUDGET_MIN` is used up (or you press Stop -- it still
saves). Every `CHECKPOINT_EVERY_MIN` minutes it saves a checkpoint, keeps
the newest two, and (main mode) pushes them to `legallm-checkpoints-b`.

**Live progress:** tqdm bar over the time budget, with timestamp, step,
loss, LR, tokens seen (the standard for every training notebook).

**fp16 safety:** steps where the loss or gradients aren't finite are
skipped by the GradScaler (normal for a handful of early steps while it
finds its scale). The run only aborts -- after saving -- if 25 steps in a
row are bad.


In [ ]:
from tqdm.auto import tqdm

run_id = f"{ARCH_ID}_{'smoke' if SMOKE_MODE else 'main'}_{time.strftime('%Y%m%d_%H%M%S')}"
print(f"run_id = {run_id}")


def training_extras():
    return {
        "schedule": SCHEDULE,
        "data_state": {"file_positions": dict(train_dataset.positions)},
        "run_mode": "smoke" if SMOKE_MODE else "main",
    }


def save_now():
    path = checkpoint_path_for(step, tokens_seen)
    save_checkpoint(path, model, optimizer, None, scaler, step, tokens_seen, MODEL_B_CONFIG, extra=training_extras())
    prune_old_checkpoints(keep=2)
    return path


model.train()
data_iter = iter(train_loader)
step, tokens_seen = start_step, start_tokens_seen
t_start = last_ckpt_time = time.time()
bad_streak, skipped_steps = 0, 0
loss_history = []
stop_reason = "unknown"

progress_bar = tqdm(total=round(TIME_BUDGET_MIN, 1), unit="min", desc=run_id, dynamic_ncols=True)
last_reported_min = 0.0
try:
    while True:
        elapsed_min = (time.time() - t_start) / 60
        if elapsed_min >= TIME_BUDGET_MIN:
            stop_reason = "time budget reached"
            break

        lr = get_lr(step)
        for group in optimizer.param_groups:
            group["lr"] = lr

        optimizer.zero_grad(set_to_none=True)
        ce_sum = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            x, y = next(data_iter)
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            out = train_model(x, targets=y).mean(0)       # [cross-entropy, loss incl. z-loss]
            scaler.scale(out[1] / GRAD_ACCUM_STEPS).backward()
            ce_sum += out[0].item()
            tokens_seen += x.numel()
        ce = ce_sum / GRAD_ACCUM_STEPS

        scaler.unscale_(optimizer)
        grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM))
        scaler.step(optimizer)          # skipped automatically if grads are inf/nan
        scaler.update()

        if math.isfinite(ce) and math.isfinite(grad_norm):
            bad_streak = 0
            loss_history.append(ce)
        else:
            bad_streak += 1
            skipped_steps += 1
            if bad_streak >= 25:
                stop_reason = "25 non-finite steps in a row"
                break

        if step % 20 == 0 or step == start_step:
            print(f"  step {step:6d}  loss={ce:.4f}  lr={lr:.2e}  gnorm={grad_norm:.2f}  "
                  f"tokens_seen={tokens_seen:,}  elapsed={elapsed_min:.1f} min")

        now_min = (time.time() - t_start) / 60
        progress_bar.update(now_min - last_reported_min)
        last_reported_min = now_min
        progress_bar.set_postfix({
            "time": datetime.datetime.now().strftime("%H:%M:%S"),
            "step": step,
            "loss": f"{ce:.3f}",
            "lr": f"{lr:.1e}",
            "tokens": f"{tokens_seen:,}",
        })

        step += 1
        if (time.time() - last_ckpt_time) / 60 >= CHECKPOINT_EVERY_MIN:
            path = save_now()
            print(f"  [checkpoint saved -> {path.name}]")
            if PUSH_DURING_TRAINING:
                push_checkpoint_dataset(f"{run_id}: step={step}, tokens_seen={tokens_seen:,} (in progress)")
            last_ckpt_time = time.time()
except KeyboardInterrupt:
    stop_reason = "stopped by user"
finally:
    progress_bar.close()

final_path = save_now()
total_min = (time.time() - t_start) / 60
print(f"\nStopped ({stop_reason}) at step={step:,}, tokens_seen={tokens_seen:,}, after {total_min:.1f} min "
      f"(finished {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}). Skipped steps: {skipped_steps}.")
print(f"Final checkpoint -> {final_path.name}")

# Round-trip check: the file on disk really contains these weights.
_reloaded = load_checkpoint(final_path, map_location="cpu")
_same = all(torch.equal(_reloaded["model_state"][k], v.detach().cpu()) for k, v in model.state_dict().items())
assert _same and _reloaded["step"] == step, "Saved checkpoint does not match the model in memory!"
print("Checkpoint round-trip check passed.")
del _reloaded
gc.collect()

if stop_reason == "25 non-finite steps in a row":
    raise RuntimeError("Training diverged (checkpoint saved). Lower PEAK_LR and resume.")


**Manual download link** -- a backup that doesn't depend on the
`KAGGLE_API_TOKEN` push working.


In [ ]:
from IPython.display import FileLink, display

print("Download the final checkpoint directly (click the link below):")
display(FileLink(str(final_path.relative_to(Path.cwd())) if final_path.is_relative_to(Path.cwd()) else str(final_path)))
print(f"({final_path.stat().st_size / 1e6:.1f} MB -- model + optimizer state.)")


## 11. Validation loss and bits-per-byte on the dev set

Scores the model on exactly what the task asks: the **continuation**
(`dev_truth`) given the end of the prompt. Only the continuation tokens
count. Also reports **bits-per-byte** (the `runs.csv` column), which --
unlike per-token loss -- is comparable across tokenizers (T1 vs T2 models).


In [ ]:
with open(INPUT_DATA_DIR / "dev_prompts.json") as f:
    dev_prompts = json.load(f)
with open(INPUT_DATA_DIR / "dev_truth.json") as f:
    dev_truth = json.load(f)
assert isinstance(dev_prompts, dict) and isinstance(dev_truth, dict), "Expected {id: {\"text\": ...}} objects."
dev_ids = sorted(set(dev_prompts) & set(dev_truth))
print(f"Loaded {len(dev_prompts)} dev prompts, {len(dev_truth)} truth records, {len(dev_ids)} shared IDs.")

model.eval()
total_nll, total_tokens, total_bytes = 0.0, 0, 0
with torch.no_grad():
    for doc_id in dev_ids[:200]:
        cont_text = " " + dev_truth[doc_id]["text"].strip()
        cont_ids = tokenizer.encode(cont_text).ids
        prompt_ids = tokenizer.encode(dev_prompts[doc_id]["text"]).ids
        room = TRAIN_SEQ_LEN - len(cont_ids)
        if room < 16 or not cont_ids:
            continue
        ids = prompt_ids[-room:] + cont_ids
        ids_t = torch.tensor([ids], device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(ids_t)
        n_ctx = len(ids) - len(cont_ids)
        nll = F.cross_entropy(logits[0, n_ctx - 1:-1].float(), ids_t[0, n_ctx:], reduction="sum")
        total_nll += nll.item()
        total_tokens += len(cont_ids)
        total_bytes += len(cont_text.encode("utf-8"))
model.train()

val_loss = total_nll / total_tokens if total_tokens else float("nan")
bits_per_byte = total_nll / (math.log(2) * total_bytes) if total_bytes else float("nan")
print(f"Dev continuation loss: {val_loss:.4f} (per token)   bits-per-byte: {bits_per_byte:.4f}")


## 12. Generate predictions

Same dev IDs as Model A (first 100 by sorted ID) so the two are directly
comparable. Two cheap output-hygiene fixes over A:
- **Length:** the references are ~128 *words*; T1 averages ~1.7 tokens per
  word, so 128 *tokens* is only ~75 words (hurts ROUGE recall and BLEU's
  brevity penalty). B generates up to 224 tokens and cuts to 128 words.
- **Repetition:** a mild repetition penalty (1.1) against the greedy
  looping small models are prone to.

Generation is slow (one token at a time): ~100 examples here; widen only
when making a real submission.


In [ ]:
GENERATION_SAMPLE_SIZE = 10 if SMOKE_MODE else min(100, len(dev_ids))
MAX_NEW_TOKENS = 224
TARGET_WORDS = 128
REPETITION_PENALTY = 1.1
sample_ids = dev_ids[:GENERATION_SAMPLE_SIZE]

predictions = {}
t0 = time.time()
for doc_id in tqdm(sample_ids, desc="generating", dynamic_ncols=True):
    prompt_ids = tokenizer.encode(dev_prompts[doc_id]["text"]).ids
    prompt_ids = prompt_ids[-(MODEL_B_CONFIG["max_seq_len"] - MAX_NEW_TOKENS):]
    prompt_tensor = torch.tensor([prompt_ids], device=DEVICE)
    out_ids = model.generate(prompt_tensor, max_new_tokens=MAX_NEW_TOKENS, kv_cache=True,
                             sampler="greedy", repetition_penalty=REPETITION_PENALTY)
    words = tokenizer.decode(out_ids[0, len(prompt_ids):].tolist()).split()[:TARGET_WORDS]
    predictions[doc_id] = {"text": " ".join(words) if words else "."}   # evaluator rejects empty text

elapsed = time.time() - t0
print(f"Generated {len(predictions)} predictions in {elapsed:.1f}s ({elapsed / max(1, len(predictions)):.2f}s/example).")
print("\nFirst prediction:", json.dumps({sample_ids[0]: predictions[sample_ids[0]]}, indent=2)[:400])

predictions_path = OUTPUT_DIR / "dev_predictions.json"
with open(predictions_path, "w") as f:
    json.dump(predictions, f, indent=2)
truth_subset_path = OUTPUT_DIR / "dev_truth_subset.json"
with open(truth_subset_path, "w") as f:
    json.dump({doc_id: dev_truth[doc_id] for doc_id in sample_ids}, f, indent=2)
print(f"\nSaved -> {predictions_path}\nSaved -> {truth_subset_path}")


## 13. Score locally with evaluate_phase1.py's own metrics

Same algorithms as the official `evaluate_phase1.py` (BERTScore F1,
ROUGE-L F1, BLEU-4, exact ID matching) -- identical to Model A's Section 13.


In [ ]:
from collections import Counter

try:
    import bert_score
except ImportError:
    print("Installing bert-score (one-time, this session)...")
    subprocess.run(["pip", "install", "-q", "bert-score"], check=True)
    import bert_score


def validate_eval_format(data, name):
    if not isinstance(data, dict) or not data:
        raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for doc_id, rec in data.items():
        if (not isinstance(doc_id, str) or not doc_id.strip() or not isinstance(rec, dict)
                or "text" not in rec or not isinstance(rec["text"], str) or not rec["text"].strip()):
            raise ValueError(f"Invalid record for ID {doc_id!r} in {name}; each record needs a non-empty string field 'text'.")


def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count() if hasattr(int, "bit_count") else bin(state).count("1")


def rouge_l_f1(pred_text, ref_text):
    pt, rt = pred_text.split(), ref_text.split()
    if not pt or not rt:
        return 0.0
    l = lcs_length(pt, rt)
    if not l:
        return 0.0
    precision, recall = l / len(pt), l / len(rt)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    order = 4
    clipped, totals = [0] * order, [0] * order
    pred_len, ref_len = 0, 0
    for pred_tokens, ref_tokens in zip(preds, refs):
        pred_len += len(pred_tokens)
        ref_len += len(ref_tokens)
        for n in range(1, order + 1):
            pred_ngrams, ref_ngrams = ngrams(pred_tokens, n), ngrams(ref_tokens, n)
            totals[n - 1] += sum(pred_ngrams.values())
            for gram, count in pred_ngrams.items():
                clipped[n - 1] += min(count, ref_ngrams.get(gram, 0))
    precisions = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(p <= 0 for p in precisions) or pred_len == 0:
        return 0.0
    brevity_penalty = 1.0 if pred_len > ref_len else math.exp(1.0 - ref_len / pred_len)
    return brevity_penalty * math.exp(sum(math.log(p) for p in precisions) / order)


def score_predictions(pred_dict, truth_dict):
    validate_eval_format(truth_dict, "truth")
    validate_eval_format(pred_dict, "prediction")
    if set(truth_dict) != set(pred_dict):
        missing = len(set(truth_dict) - set(pred_dict))
        extra = len(set(pred_dict) - set(truth_dict))
        raise ValueError(f"ID mismatch: missing={missing}, extra={extra}")
    ordered_ids = list(truth_dict)
    pred_texts = [pred_dict[i]["text"].strip() for i in ordered_ids]
    ref_texts = [truth_dict[i]["text"].strip() for i in ordered_ids]
    eval_device = "cuda" if torch.cuda.is_available() else "cpu"
    _, _, f1 = bert_score.score(pred_texts, ref_texts, lang="en", batch_size=8, verbose=False, device=eval_device)
    bertscore = float(f1.mean().item())
    rouge = sum(rouge_l_f1(p, r) for p, r in zip(pred_texts, ref_texts)) / len(ordered_ids)
    b4 = bleu4([t.split() for t in pred_texts], [t.split() for t in ref_texts])
    return bertscore, rouge, b4


with open(predictions_path) as f:
    _pred_for_scoring = json.load(f)
with open(truth_subset_path) as f:
    _truth_for_scoring = json.load(f)

try:
    dev_bertscore, dev_rouge_l, dev_bleu4 = score_predictions(_pred_for_scoring, _truth_for_scoring)
    print(f"Examples:     {len(_truth_for_scoring)}")
    print(f"BERTScore F1: {dev_bertscore:.6f}")
    print(f"ROUGE-L F1:   {dev_rouge_l:.6f}")
    print(f"BLEU-4:       {dev_bleu4:.6f}")
    if SMOKE_MODE:
        print("\n(Smoke run: low scores are expected. This cell finishing without an error is the check.)")
except Exception as exc:
    dev_bertscore, dev_rouge_l, dev_bleu4 = None, None, None
    print(f"Scoring failed: {exc}")


## 14. Append this run to `runs.csv`

Same columns as the plan doc's contract. B also fills `bits_per_byte`.


In [ ]:
import csv

RUNS_CSV_PATH = OUTPUT_DIR / "runs.csv"
RUNS_CSV_COLUMNS = [
    "run_id", "arch_id", "technique", "params", "tokens_seen", "tokens_per_sec",
    "peak_mem_gb", "val_loss", "bits_per_byte", "dev_bertscore", "dev_rouge_l",
    "dev_bleu4", "codabench_score", "checkpoint_path", "notes",
]

session_tokens = tokens_seen - start_tokens_seen
tokens_per_sec = session_tokens / (total_min * 60) if total_min > 0 else None
peak_mem_gb = max(torch.cuda.max_memory_allocated(d) for d in range(GPUS_USED)) / 1e9 if N_GPUS else 0.0

row = {
    "run_id": run_id,
    "arch_id": ARCH_ID,
    "technique": "full-pretrain",
    "params": n_params,
    "tokens_seen": tokens_seen,
    "tokens_per_sec": round(tokens_per_sec, 1) if tokens_per_sec else None,
    "peak_mem_gb": round(peak_mem_gb, 2),
    "val_loss": round(val_loss, 4) if math.isfinite(val_loss) else None,
    "bits_per_byte": round(bits_per_byte, 4) if math.isfinite(bits_per_byte) else None,
    "dev_bertscore": round(dev_bertscore, 4) if dev_bertscore is not None else None,
    "dev_rouge_l": round(dev_rouge_l, 4) if dev_rouge_l is not None else None,
    "dev_bleu4": round(dev_bleu4, 4) if dev_bleu4 is not None else None,
    "codabench_score": None,
    "checkpoint_path": final_path.name,
    "notes": (f"{'smoke' if SMOKE_MODE else 'main'}; {GPUS_USED} GPU(s); seq_len={TRAIN_SEQ_LEN}; "
              f"window={MODEL_B_CONFIG['window']}; lr_decay_steps={SCHEDULE['lr_decay_steps']}; "
              f"gen={MAX_NEW_TOKENS}tok->{TARGET_WORDS}words rep={REPETITION_PENALTY}; stop={stop_reason}"),
}

file_exists = RUNS_CSV_PATH.exists()
with open(RUNS_CSV_PATH, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=RUNS_CSV_COLUMNS)
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)
print(json.dumps(row, indent=2))


## 15. Final publish

Merges the previous `runs.csv` from `legallm-checkpoints-b` (if attached),
keeps the newest two checkpoints, and pushes a new dataset version.


In [ ]:
prev_runs = [p for p in KAGGLE_INPUT.rglob("runs.csv") if CHECKPOINT_DATASET_SLUG in str(p)] if KAGGLE_INPUT.exists() else []
if prev_runs:
    with open(prev_runs[0]) as f:
        prev_rows = list(csv.DictReader(f))
    with open(RUNS_CSV_PATH) as f:
        new_rows = list(csv.DictReader(f))
    known = {r["run_id"] for r in new_rows}
    merged = [r for r in prev_rows if r["run_id"] not in known] + new_rows
    with open(RUNS_CSV_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RUNS_CSV_COLUMNS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(merged)
    print(f"Merged {len(merged) - len(new_rows)} earlier run(s) into runs.csv.")

prune_old_checkpoints(keep=2)
pushed = push_checkpoint_dataset(f"{run_id} final: step={step}, tokens_seen={tokens_seen:,}, "
                                 f"val_loss={val_loss:.4f}")
if pushed:
    print(f"Done. Attach '{CHECKPOINT_DATASET_SLUG}' as input next session to resume.")
else:
    print(f"Push failed -- use the download link in Section 10, or download {OUTPUT_DIR} from the Output panel.")


## 16. Summary

| Item | Where |
| --- | --- |
| Model self-tests (cache = full forward, window really local, loss) | Section 4 |
| Parameter count, layer pattern | Section 4 |
| 1 or 2 GPUs, seq_len, batch, tokens/s, LR schedule | Section 5 |
| Training log + final checkpoint + round-trip check | Section 10 |
| Dev continuation loss, bits-per-byte | Section 11 |
| Local BERTScore / ROUGE-L / BLEU-4 | Section 13 |
| `runs.csv` row | Section 14 |
| Checkpoints | dataset `legallm-checkpoints-b` |

**After the smoke run:** if it reached this cell with no errors, set
`SMOKE_MODE = False` and do **Save Version -> Save & Run All**. The main
run starts fresh (smoke checkpoints are never resumed).

**Next session (optional):** attach `legallm-checkpoints-b`, then Save &
Run All again. It resumes weights, optimizer, LR schedule, and data
position automatically.
